In [5]:
import torch
import torchvision.datasets as dataset
import torchvision.transforms as transforms
import torch.nn.init

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# random seed 고정
torch.manual_seed(777)

if device == "cuda":
    torch.cuda.manual_seed_all(777)

In [3]:
learning_rate = 0.001
training_epochs = 15
batch_size = 100

In [6]:
# Dataset
mnist_train = dataset.MNIST(root="MNIST_data/",
                            train=True,
                            transform=transforms.ToTensor(),
                            download=True)

mnist_test = dataset.MNIST(root="MNIST_data/",
                           train=False,
                           transform=transforms.ToTensor(),
                           download=True)

In [7]:
data_loader = torch.utils.data.DataLoader(dataset=mnist_train,
                                          batch_size=batch_size,
                                          shuffle=True,
                                          drop_last=True)

In [8]:
class CNN(torch.nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.keep_prob = 0.5 # dropout (keep probability)

        # L1: Conv Layer
        # Input: (?, 28, 28, 1)
        # Conv2d: 32 channels, 3x3 kernel, stride 1, padding 1
        # ReLU Activation
        # MaxPool2d: 2x2 kernel, stride 2 downsampling -> Output: (?, 14, 14, 32)
        self.layer1 = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(stride=2, kernel_size=2)
        )

        # L2: Conv Layer
        # Input: (?, 14, 14, 32)
        # Conv2d: 64 channels, 3x3 kernel, stride 1, padding 1
        # ReLU Activation
        # MaxPool2d: 2x2 kernel, stride 2 downsampling -> Output: (?, 7, 7, 64)
        self.layer2 = torch.nn.Sequential(
            torch.nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(stride=2, kernel_size=2)
        )

        # L3: Conv Layer
        # Input: (?, 7, 7, 64)
        # Conv2d: 128 channels, 3x3 kernel, stride 1, padding 1
        # ReLU Activation
        # MaxPool2d: 2x2 kernel, stride 2, padding 1 downsampling -> Output: (?, 4, 4, 128)
        self.layer3 = torch.nn.Sequential(
            torch.nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(stride=2, kernel_size=2, padding=1)
        )

        # L4: Fully Connected Layer
        # Input: (4, 4, 128) ->  Output: 625
        # ReLU Activation
        # Dropout p=0.5
        self.fc1 = torch.nn.Linear(4 * 4 * 128, 625, bias=True)
        torch.nn.init.xavier_uniform_(self.fc1.weight)
        self.layer4 = torch.nn.Sequential(
            self.fc1,
            torch.nn.ReLU(),
            torch.nn.Dropout(p=1 - self.keep_prob)
        )

        # L5: Fully Connected Layer
        # Input: 625 -> Output: 10
        self.fc2 = torch.nn.Linear(625, 10, bias=True)
        torch.nn.init.xavier_uniform_(self.fc2.weight)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        out = out.view(out.size(0), -1) # Tensor flatten
        out = self.layer4(out)
        out = self.fc2(out)
        return out

In [9]:
model = CNN().to(device)

In [10]:
criterion = torch.nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [11]:
total_batch = len(data_loader)
print(f"총 배치의 수 : {total_batch}")

총 배치의 수 : 600


In [17]:
for epoch in range(training_epochs):
    avg_cost = 0

    for X, Y in data_loader:
        # 이미지 데이터는 이미 28 by 28 -> 별도 reshape 불필요
        # 레이블 Y는 원-핫 인코딩이 아닌 정수형 클래스 라벨
        X = X.to(device)
        Y = Y.to(device)

        optimizer.zero_grad() # optimizer 기울기 초기화
        hypothesis = model(X) # 모델에 X 넣어 예측값 계산 forward
        cost = criterion(hypothesis, Y) # 예측값과 실제값간의 손실 계산
        cost.backward() # 역전파로 기울기 계산
        optimizer.step() # 가중치 업데이트

        avg_cost += cost / total_batch

    print(f"[Epoch: {epoch+1.:>4}] cost = {avg_cost:>.9}")

[Epoch:  1.0] cost = 0.0046081706
[Epoch:  2.0] cost = 0.00521397032
[Epoch:  3.0] cost = 0.00518768141
[Epoch:  4.0] cost = 0.0052559888
[Epoch:  5.0] cost = 0.00453671813
[Epoch:  6.0] cost = 0.00478419801
[Epoch:  7.0] cost = 0.00522329845
[Epoch:  8.0] cost = 0.00409488752
[Epoch:  9.0] cost = 0.00249766419
[Epoch: 10.0] cost = 0.00353399036
[Epoch: 11.0] cost = 0.00596542284
[Epoch: 12.0] cost = 0.00300582848
[Epoch: 13.0] cost = 0.00385404611
[Epoch: 14.0] cost = 0.00457315147
[Epoch: 15.0] cost = 0.0051928428


In [13]:
with torch.no_grad():
    X_test = mnist_test.data.view(len(mnist_test), 1, 28, 28).float().to(device) # test dataset 크기 맞추고 flatten
    Y_test = mnist_test.targets.to(device)

    prediction = model(X_test) # 모델에 테스트 데이터 넣어 예측값 계산
    correct_prediction = torch.argmax(prediction, 1) == Y_test # 예측값과 실제값 비교
    accuracy = correct_prediction.float().mean() # 정확도를 계산하기 위해 일치하는 예측의 평균
    print(f"Accuracy: {accuracy.item()}, {accuracy*100:.2f}%")

Accuracy: 0.9820999503135681, 98.21%


In [16]:
# GPU 메모리 정리
import gc
gc.collect()
torch.cuda.empty_cache()